# Episode 31: Mixture of experts from scratch in PyTorch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep031-moe.ipynb)

We replace each block's MLP in our GPT with a mixture of eight experts and a router, write it as a loop and then as one batched multiply, balance it three ways, and train it next to the dense model at the same compute per token.

[The video](https://youtu.be/TwfM0aXovPo) · [The episode's page](https://mvgpt.nenuadrian.com/ep031-moe/) · [The docs](https://nenuadrian.github.io/mvGPT/moe/) · [The code at `ep031/end`](https://github.com/nenuadrian/mvGPT/tree/ep031/end)

This notebook runs the episode's commands on its code, as the episode left it. What you train here comes out a little different from the video's, as training runs for a set time, on another machine. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep031/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")

## The tests
The mixture, sorted by expert and run as one batched multiply, against a loop over the tokens;
an expert's weights times top k against the MLP's; the balancing loss at both of its ends; and
the bias, which moves the choice of experts and leaves their weights to the router's scores.

In [ ]:
!python -m pytest -q tests/test_moe.py

## A step of each, timed
The dense model and the mixture of eight experts, a training step each on random tokens. On the
Mac, the mixture's step took 1.17 times as long.

In [ ]:
!python moe_bench.py

## The data

In [ ]:
!python data.py  # TinyStories, the 10,000-word vocabulary, and the stories as token ids

## Four runs
The episode trained each for 2,000 steps, 20 to 36 minutes apiece on the Mac: the modern block
dense, with eight experts balanced by a loss, with no balancing, and with DeepSeek-V3's bias.
Here they get 1,000 steps each, to `checkpoints/pretrain-modern`, `pretrain-modern-moe8`,
`pretrain-modern-moe8-none` and `pretrain-modern-moe8-bias`. The mixtures' lines end with
their experts' smallest and largest share of the tokens, where an even split is 12.5%.

In [ ]:
!python pretrain.py --modern --steps 1000
!python pretrain.py --modern --moe 8 --steps 1000
!python pretrain.py --modern --moe 8 --balance none --steps 1000
!python pretrain.py --modern --moe 8 --balance bias --steps 1000

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## What each expert gets
Each block's first choices by expert, the words each expert gets most, with the share of their
uses that went to it, and a held-out story with each word in its expert's colour. In the
video's balanced model, block 3 split the words by kind: the ones that start a clause to one
expert, the punctuation to another, the nouns to a third. Without balancing, some experts get
next to nothing.

In [ ]:
!python experts.py checkpoints/pretrain-modern-moe8/model.pt
!python experts.py checkpoints/pretrain-modern-moe8-none/model.pt

Next: [episode 33, Sparse autoencoder features inside our GPT](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep033-sae.ipynb)